##Step 01 - Build RAG Pipeline

In [ ]:
# ----- Azure AI Search index: create or update -----
credential  = AzureKeyCredential(AZURE_SEARCH_API_KEY)
idx_client  = SearchIndexClient(endpoint=AZURE_SEARCH_ENDPOINT, credential=credential)
srch_client = SearchClient(
    endpoint=AZURE_SEARCH_ENDPOINT,
    index_name=AZURE_SEARCH_INDEX_NAME,
    credential=credential,
)

fields = [
    SimpleField(name="id",       type=SearchFieldDataType.String, key=True),
    SearchableField(name="content",  type=SearchFieldDataType.String,
                    analyzer_name="en.microsoft"),
    SimpleField(name="title",    type=SearchFieldDataType.String, filterable=True),
    SimpleField(name="url",      type=SearchFieldDataType.String),
    SearchField(
        name="embedding",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=EMBEDDING_DIM,
        vector_search_profile_name="hnsw-profile",
    ),
]

vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="hnsw-algo")],
    profiles=[VectorSearchProfile(
        name="hnsw-profile",
        algorithm_configuration_name="hnsw-algo",
    )],
)

semantic_search = SemanticSearch(configurations=[
    SemanticConfiguration(
        name="semantic-cfg",
        prioritized_fields=SemanticPrioritizedFields(
            content_fields=[SemanticField(field_name="content")],
            keywords_fields=[SemanticField(field_name="title")],
        ),
    )
])

index_def = SearchIndex(
    name=AZURE_SEARCH_INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search,
)

result = idx_client.create_or_update_index(index_def)
print(f"Index '{result.name}' ready (hybrid: BM25 + HNSW vector)")

Index 'hybrid-rag-longcontext384' ready (hybrid: BM25 + HNSW vector)


In [ ]:
BATCH = 100
print(f"Uploading {len(all_docs)} documents to Azure AI Search...")
t0 = time.perf_counter()

for i in tqdm(range(0, len(all_docs), BATCH), desc="Azure upload"):
    batch_end = min(i + BATCH, len(all_docs))
    srch_client.upload_documents(documents=[
        {
            "id":        f"doc-{i + j}",
            "content":   texts[i + j],
            "title":     metas[i + j].get("title", ""),
            "url":       metas[i + j].get("url", ""),
            "embedding": all_vectors[i + j],
        }
        for j in range(batch_end - i)
    ])

print(f"Uploaded {len(all_docs)} documents in {time.perf_counter()-t0:.1f}s")

Uploading 40 documents to Azure AI Search...


Azure upload:   0%|          | 0/1 [00:00<?, ?it/s]

Uploaded 40 documents in 1.0s


In [ ]:
def azure_hybrid_search(query: str, top_k: int = TOP_K) -> list:
    """BM25 keyword + HNSW vector search, fused with Reciprocal Rank Fusion (RRF)."""
    query_vec = embedder.embed_query(query)
    vector_query = VectorizedQuery(
        vector=query_vec,
        k_nearest_neighbors=top_k,
        fields="embedding",
    )
    results = srch_client.search(
        search_text=query,
        vector_queries=[vector_query],
        select=["id", "content", "title", "url"],
        top=top_k,
    )
    return list(results)

In [ ]:
# Build LangChain FAISS vectorstore from pre-computed vectors.
# FAISS.from_embeddings() reuses all_vectors -- zero extra API calls.
text_embeddings = list(zip(texts, all_vectors))  # [(text, vector), ...]

faiss_store = FAISS.from_embeddings(
    text_embeddings=text_embeddings,
    embedding=embedder,   # used only for embed_query() at search time
    metadatas=metas,
)
print(f"FAISS vectorstore ready ({len(texts)} vectors, no re-embedding)")

# Claude LLM
llm = ChatAnthropic(model=CLAUDE_MODEL, max_tokens=512)
print(f"ChatAnthropic ready: {CLAUDE_MODEL}")

FAISS vectorstore ready (40 vectors, no re-embedding)
ChatAnthropic ready: claude-sonnet-4-6


In [ ]:
%%capture
!pip install langchain_groq

In [ ]:
# ============================================================
# RAG Generation using Groq
# ============================================================

from dataclasses import dataclass
import time

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

# ------------------------------------------------------------
# Configure Groq
# ------------------------------------------------------------

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0
)

# Groq pricing (optional)
COST_INPUT = 0.59 / 1_000_000
COST_OUTPUT = 0.79 / 1_000_000

# ------------------------------------------------------------
# Result Dataclass
# ------------------------------------------------------------

@dataclass
class RAGResult:
    answer: str
    sources: list
    embed_ms: float
    retrieve_ms: float
    generate_ms: float
    input_tokens: int = 0
    output_tokens: int = 0

    @property
    def total_ms(self):
        return (
            self.embed_ms
            + self.retrieve_ms
            + self.generate_ms
        )

    @property
    def cost_usd(self):
        return (
            self.input_tokens * COST_INPUT
            + self.output_tokens * COST_OUTPUT
        )

# ------------------------------------------------------------
# Main RAG Function
# ------------------------------------------------------------

def rag_answer(query: str) -> RAGResult:

    # --------------------------------------------------------
    # Stage 1: Query Embedding
    # --------------------------------------------------------

    t0 = time.perf_counter()

    _ = embedder.embed_query(query)

    embed_ms = (
        time.perf_counter() - t0
    ) * 1000

    # --------------------------------------------------------
    # Stage 2: Azure Hybrid Retrieval
    # --------------------------------------------------------

    t0 = time.perf_counter()

    raw = azure_hybrid_search(query)

    retrieve_ms = (
        time.perf_counter() - t0
    ) * 1000

    context = "\n\n".join(
        [
            f"[{r['title']}]\n{r['content']}"
            for r in raw
        ]
    )

    sources = list(
        {
            r["title"]
            for r in raw
        }
    )

    # --------------------------------------------------------
    # Stage 3: Groq Generation
    # --------------------------------------------------------

    messages = [
        SystemMessage(
            content=(
                "Answer ONLY from the provided context. "
                "If the answer is not present in the context, "
                "say that the information is unavailable. "
                "Be concise."
            )
        ),
        HumanMessage(
            content=f"{context}\n\nQuestion: {query}"
        )
    ]

    t0 = time.perf_counter()

    response = llm.invoke(messages)

    generate_ms = (
        time.perf_counter() - t0
    ) * 1000

    # --------------------------------------------------------
    # Token Usage
    # --------------------------------------------------------

    usage = response.response_metadata.get(
        "token_usage",
        {}
    )

    return RAGResult(
        answer=response.content,
        sources=sources,
        embed_ms=embed_ms,
        retrieve_ms=retrieve_ms,
        generate_ms=generate_ms,
        input_tokens=usage.get(
            "prompt_tokens",
            0
        ),
        output_tokens=usage.get(
            "completion_tokens",
            0
        )
    )

# ------------------------------------------------------------
# Smoke Test
# ------------------------------------------------------------

test = rag_answer(
    "What is retrieval-augmented generation?"
)

print("RAG pipeline smoke test passed!")
print()

print("Sources:")
print(test.sources)

print()

print(
    f"Embed     : {test.embed_ms:.0f} ms"
)
print(
    f"Retrieve  : {test.retrieve_ms:.0f} ms"
)
print(
    f"Generate  : {test.generate_ms:.0f} ms"
)
print(
    f"Total     : {test.total_ms:.0f} ms"
)

print()

print(
    f"Tokens In : {test.input_tokens}"
)
print(
    f"Tokens Out: {test.output_tokens}"
)

print(
    f"Cost      : ${test.cost_usd:.6f}"
)

print("\nAnswer:")
print(test.answer)

RAG pipeline smoke test passed!

Sources:
['BM25 Information Retrieval', 'Transformer Architecture', 'Retrieval-Augmented Generation']

Embed     : 61 ms
Retrieve  : 590 ms
Generate  : 562 ms
Total     : 1212 ms

Tokens In : 420
Tokens Out: 45
Cost      : $0.000283

Answer:
Retrieval-Augmented Generation (RAG) combines a retrieval system with a generative language model, retrieving relevant documents from an external knowledge base at inference time to ground generation in verifiable facts and reduce hallucination.
